# Flight Delay Regression Training

This notebook trains regression models to predict delay minutes using the labeled dataset produced by the data pipeline.
It follows the modeling requirements: chronological split, leakage-safe preprocessing, baseline comparison, and model artifacts.

- Filtered `minutes_to_departure_at_snapshot >= 0` to prevent Data Leakage.
- Added Cyclical Encoding for `scheduled_hour`.
- Added Binning for `visibility_miles`.
- Dropped `temperature_c` to avoid multicollinearity.
- Added `class_weight='balanced'` to Two-Stage Classifier.
- TransformedTargetRegressor with `np.log1p` applied to Regressors.
- Added Permutation Feature Importance for Two-Stage Model.
- Used TimeSeriesSplit for Optuna tuning.

In [1]:
from __future__ import annotations

from pathlib import Path
import math
import warnings

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import TimeSeriesSplit

warnings.filterwarnings("ignore")

In [2]:
data_path = Path("..") / "Data Collection + Processing" / "data" / "processed" / "training_dataset_labeled.csv"
df = pd.read_csv(data_path)
df.head()

,retrieved_at_vn,flight_date,direction,scheduled_time,estimated_time,route_airport,flight_number,status_raw,source_airport,route_airport_std,...,is_wind_variable,scheduled_hour,scheduled_dayofweek,scheduled_month,airline_code,flight_num_only,minutes_to_departure_at_snapshot,is_estimated_missing,temp_dew_spread,is_low_visibility
0,2026-04-09 23:39:16,2026-04-09,Arrival,07:05,06:57,Hà Nội (HAN),QH101,Đúng giờ,DN,HA NOI (HAN),...,1,7,3,4,QH,101,-994.266667,0,4,0.0
1,2026-04-09 23:39:16,2026-04-09,Departure,13:40,13:40,TP. Hồ Chí Minh (SGN),VN129,3-16,DN,HO CHI MINH (SGN),...,1,13,3,4,VN,129,-599.266667,0,4,0.0
2,2026-04-09 23:39:16,2026-04-09,Departure,13:20,13:40,Hà Nội (HAN),9G922,35-39,DN,HA NOI (HAN),...,1,13,3,4,9G,9,-619.266667,0,4,0.0
3,2026-04-09 23:39:16,2026-04-09,Departure,13:20,13:20,Cần Thơ (VCA),VJ703,27-34,DN,CAN THO (VCA),...,1,13,3,4,VJ,703,-619.266667,0,4,0.0
4,2026-04-09 23:39:16,2026-04-09,Departure,13:15,13:15,Hà Nội (HAN),VJ510,27-34,DN,HA NOI (HAN),...,1,13,3,4,VJ,510,-624.266667,0,4,0.0


## Basic cleanup, Target capping, and Fix Data Leakage
- Filter `minutes_to_departure_at_snapshot >= 0` to ensure predictions are made pre-flight.
- Parse `scheduled_dt` and sort for chronological split.
- Cap `delay_minutes` to [0, 300] to reduce extreme noise.

In [3]:
df["scheduled_dt"] = pd.to_datetime(df["scheduled_dt"], errors="coerce")
df = df.dropna(subset=["scheduled_dt", "delay_minutes"]).copy()

# Fix Data Leakage: Only use snapshots taken BEFORE departure
df = df[df["minutes_to_departure_at_snapshot"] >= 0].copy()

df = df.sort_values("scheduled_dt").reset_index(drop=True)

df["delay_minutes_raw"] = df["delay_minutes"]
df["delay_minutes"] = df["delay_minutes"].clip(lower=0, upper=300)

df[["delay_minutes_raw", "delay_minutes"]].describe()

,delay_minutes_raw,delay_minutes
count,237.000000,237.000000
mean,1.278481,2.645570
std,10.407303,9.158729
min,-25.000000,0.000000
25%,0.000000,0.000000
50%,0.000000,0.000000
75%,0.000000,0.000000
max,90.000000,90.000000


## Feature Engineering & Selection
- **Cyclical features**: `sin_hour` and `cos_hour`.
- **Visibility binning**: Bad (<3), Medium (3-6), Good (>6).
- **Multicollinearity**: Keep `temp_dew_spread`, drop `temperature_c`.

In [4]:
# Cyclical Encoding
df["sin_hour"] = np.sin(2 * np.pi * df["scheduled_hour"] / 24)
df["cos_hour"] = np.cos(2 * np.pi * df["scheduled_hour"] / 24)

# Visibility Binning
df["visibility_bin"] = pd.cut(
    df["visibility_miles"],
    bins=[-np.inf, 3, 6, np.inf],
    labels=["Bad", "Medium", "Good"]
).astype(str)

feature_cols = [
    "scheduled_hour", # Kept for tree-models, though sin/cos is also available
    "sin_hour",
    "cos_hour",
    "scheduled_dayofweek",
    "scheduled_month",
    "minutes_to_departure_at_snapshot",
    "source_airport",
    "direction",
    "route_airport_std",
    "flight_number",
    "airline_code",
    "flight_num_only",
    "is_estimated_missing",
    "dew_point_c",
    "wind_direction_deg",
    "wind_speed_kt",
    "visibility_miles",
    "visibility_bin",
    "cloud_cover",
    "temp_dew_spread",
    "is_wind_variable",
]

missing_cols = [c for c in feature_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing columns: {missing_cols}")

X = df[feature_cols].copy()
y = df["delay_minutes"].copy()

X.head()

,scheduled_hour,sin_hour,cos_hour,scheduled_dayofweek,scheduled_month,minutes_to_departure_at_snapshot,source_airport,direction,route_airport_std,flight_number,...,flight_num_only,is_estimated_missing,dew_point_c,wind_direction_deg,wind_speed_kt,visibility_miles,visibility_bin,cloud_cover,temp_dew_spread,is_wind_variable
0,23,-0.258819,0.965926,3,4,4.783333,NB,Arrival,PHU QUOC,9G1242,...,9,0,25,100.0,7,3.73,Medium,"FEW@1100ft, BKN@2000ft",2,0
1,19,-0.965926,0.258819,6,4,6.116667,TSN,Arrival,PLEIKU,VJ395,...,395,0,24,160.0,10,6.00,Medium,clear,6,0
2,19,-0.965926,0.258819,6,4,6.116667,TSN,Departure,PLEIKU,VJ395,...,395,0,24,160.0,10,6.00,Medium,clear,6,0
3,19,-0.965926,0.258819,6,4,7.716667,DN,Departure,HA NOI (HAN),VJ520,...,520,0,25,80.0,4,6.00,Medium,FEW@2000ft,3,0
4,19,-0.965926,0.258819,6,4,7.716667,DN,Departure,HO CHI MINH (SGN),VN7107,...,7107,0,25,80.0,4,6.00,Medium,FEW@2000ft,3,0


## Chronological split
Using row-index based chronological split since dates might be sparse after filtering out data leakage.

In [5]:
n = len(df)
train_end = int(n * 0.7)
val_end = int(n * 0.85)

X_train, y_train = X.iloc[:train_end], y.iloc[:train_end]
X_val, y_val = X.iloc[train_end:val_end], y.iloc[train_end:val_end]
X_test, y_test = X.iloc[val_end:], y.iloc[val_end:]

print((X_train.shape, X_val.shape, X_test.shape))

((165, 21), (36, 21), (36, 21))


## Preprocessing pipeline

In [6]:
categorical_cols = [
    "source_airport",
    "direction",
    "route_airport_std",
    "flight_number",
    "airline_code",
    "cloud_cover",
    "visibility_bin",
]

numeric_cols = [c for c in feature_cols if c not in categorical_cols]

try:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse=False)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", ohe),
    ]
)

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

preprocess = ColumnTransformer(
    transformers=[
        ("cat", categorical_transformer, categorical_cols),
        ("num", numeric_transformer, numeric_cols),
    ],
    remainder="drop",
)

## Baseline model

In [7]:
def evaluate_regression(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = math.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    return {"mae": mae, "rmse": rmse, "r2": r2}

baseline = DummyRegressor(strategy="median")
baseline.fit(X_train, y_train)
baseline_pred = baseline.predict(X_val)
baseline_metrics = evaluate_regression(y_val, baseline_pred)
print("Baseline Metrics:", baseline_metrics)

Baseline Metrics: {'mae': 1.6666666666666667, 'rmse': 7.905694150420948, 'r2': -0.046511627906976605}


## Candidate models

In [8]:
# Wrap Regressors with Log1p Transform to handle right-skewed target
def get_log_regressor(regressor):
    return TransformedTargetRegressor(
        regressor=regressor,
        func=np.log1p,
        inverse_func=np.expm1,
    )

models = {
    "RandomForest": get_log_regressor(RandomForestRegressor(
        n_estimators=100, # Reduced for faster training
        max_depth=6,
        min_samples_split=4,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1,
    ))
}

try:
    from sklearn.ensemble import HistGradientBoostingRegressor
    models["HistGB_Log1p"] = get_log_regressor(HistGradientBoostingRegressor(
        learning_rate=0.05,
        max_depth=6,
        max_iter=100,
        random_state=42,
    ))
except Exception:
    pass

try:
    from xgboost import XGBRegressor
    models["XGBoost"] = get_log_regressor(XGBRegressor(
        n_estimators=100,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=42,
    ))
except Exception:
    pass

try:
    from lightgbm import LGBMRegressor
    models["LightGBM"] = get_log_regressor(LGBMRegressor(
        n_estimators=100,
        learning_rate=0.05,
        num_leaves=31,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        verbose=-1,
    ))
except Exception:
    pass

try:
    from catboost import CatBoostRegressor
    models["CatBoost"] = get_log_regressor(CatBoostRegressor(
        depth=6,
        learning_rate=0.08,
        iterations=100,
        loss_function="RMSE",
        random_seed=42,
        verbose=False,
    ))
except Exception:
    pass

print("Models:", models.keys())

Models: dict_keys(['RandomForest', 'HistGB_Log1p', 'XGBoost', 'LightGBM', 'CatBoost'])


In [9]:
results = []
trained = {}

for name, model in models.items():
    pipe = Pipeline([
        ("preprocess", preprocess),
        ("model", model),
    ])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_val)
    metrics = evaluate_regression(y_val, pred)
    metrics["model"] = name
    metrics["baseline_mae"] = baseline_metrics["mae"]
    results.append(metrics)
    trained[name] = pipe

results_df = pd.DataFrame(results).sort_values("mae")
print(results_df)

        mae      rmse        r2         model  baseline_mae
4  1.734373  7.661487  0.017143      CatBoost      1.666667
0  1.770294  7.877868 -0.039158  RandomForest      1.666667
3  1.815817  7.710259  0.004590      LightGBM      1.666667
1  1.824861  7.792435 -0.016741  HistGB_Log1p      1.666667
2  1.837786  7.910929 -0.047898       XGBoost      1.666667


## Two-stage model (delay probability + magnitude)
Uses `class_weight='balanced'` for Classifier to address 85/15 imbalance.
Uses `TransformedTargetRegressor` with `np.log1p` for Regressor.

In [10]:
from sklearn.base import clone
from sklearn.ensemble import HistGradientBoostingClassifier

def fit_two_stage(X_train, y_train, X_val, y_val):
    y_train_pos = (y_train > 0).astype(int)

    # Stage 1: Classifier with class_weight='balanced'
    clf = HistGradientBoostingClassifier(
        learning_rate=0.05,
        max_depth=6,
        max_iter=100,
        random_state=42,
        class_weight='balanced'
    )
    clf_pipe = Pipeline([
        ("preprocess", clone(preprocess)),
        ("model", clf),
    ])
    clf_pipe.fit(X_train, y_train_pos)

    # Stage 2: Regressor with Log1p Transform
    base_reg = HistGradientBoostingRegressor(
        learning_rate=0.05,
        max_depth=6,
        max_iter=100,
        random_state=42,
    )
    reg = TransformedTargetRegressor(
        regressor=base_reg,
        func=np.log1p,
        inverse_func=np.expm1,
    )
    reg_pipe = Pipeline([
        ("preprocess", clone(preprocess)),
        ("model", reg),
    ])
    
    pos_mask = y_train > 0
    if pos_mask.sum() > 0:
        reg_pipe.fit(X_train[pos_mask], y_train[pos_mask])
    else:
        # Fallback if no positive cases
        reg_pipe.fit(X_train, y_train)

    val_prob = clf_pipe.predict_proba(X_val)[:, 1]
    val_reg = reg_pipe.predict(X_val)

    thresholds = np.linspace(0.1, 0.9, 17)
    best_threshold = 0.5
    best_mae = float("inf")
    best_pred = None
    for threshold in thresholds:
        val_pred = np.where(val_prob >= threshold, val_reg, 0.0)
        mae = mean_absolute_error(y_val, val_pred)
        if mae < best_mae:
            best_mae = mae
            best_threshold = threshold
            best_pred = val_pred

    metrics = evaluate_regression(y_val, best_pred)
    metrics["model"] = "TwoStage"
    metrics["baseline_mae"] = baseline_metrics["mae"]
    return clf_pipe, reg_pipe, best_threshold, metrics

two_stage_clf, two_stage_reg, two_stage_threshold, two_stage_metrics = fit_two_stage(
    X_train, y_train, X_val, y_val
)
print("Two Stage Metrics:", two_stage_metrics)

Two Stage Metrics: {'mae': 1.6666666666666667, 'rmse': 6.353155006656915, 'r2': 0.3241614756325799, 'model': 'TwoStage', 'baseline_mae': 1.6666666666666667}


## Select best model and evaluate on test

In [11]:
best_model_name = results_df.iloc[0]["model"]
best_pipe = trained[best_model_name]
best_mae = results_df.iloc[0]["mae"]

if "two_stage_metrics" in globals() and two_stage_metrics["mae"] < best_mae:
    best_model_name = "TwoStage"
    best_mae = two_stage_metrics["mae"]

X_trainval = pd.concat([X_train, X_val])
y_trainval = pd.concat([y_train, y_val])

if best_model_name == "TwoStage":
    y_trainval_pos = (y_trainval > 0).astype(int)
    two_stage_clf.fit(X_trainval, y_trainval_pos)
    if (y_trainval > 0).sum() > 0:
        two_stage_reg.fit(X_trainval[y_trainval > 0], y_trainval[y_trainval > 0])
    else:
        two_stage_reg.fit(X_trainval, y_trainval)
        
    test_prob = two_stage_clf.predict_proba(X_test)[:, 1]
    test_reg = two_stage_reg.predict(X_test)
    test_pred = np.where(test_prob >= two_stage_threshold, test_reg, 0.0)
    test_metrics = evaluate_regression(y_test, test_pred)
else:
    best_pipe.fit(X_trainval, y_trainval)
    test_pred = best_pipe.predict(X_test)
    test_metrics = evaluate_regression(y_test, test_pred)

print("Best Model:", best_model_name)
print("Test Metrics:", test_metrics)

Best Model: TwoStage
Test Metrics: {'mae': 2.8165773353230117, 'rmse': 5.9748628006297135, 'r2': 0.0}


## Feature importance (Permutation Importance)

In [12]:
from sklearn.inspection import permutation_importance
import matplotlib.pyplot as plt
import seaborn as sns

def get_feature_names(preprocess, categorical_cols, numeric_cols):
    cat_features = preprocess.named_transformers_["cat"].get_feature_names_out(categorical_cols)
    return np.concatenate([cat_features, numeric_cols])

if best_model_name == "TwoStage":
    print("Computing Permutation Feature Importance for Two-Stage Classifier...")
    y_val_pos = (y_val > 0).astype(int)
    # Use neg_brier_score or accuracy as scoring, by default permutation_importance uses the estimator's default score
    result = permutation_importance(
        two_stage_clf, X_val, y_val_pos, n_repeats=5, random_state=42, n_jobs=-1
    )
    
    importances = pd.Series(result.importances_mean, index=X_val.columns)
    top_importances = importances.sort_values(ascending=False).head(20)
    print("Top Importances:")
    print(top_importances)
else:
    print("Computing Permutation Feature Importance...")
    result = permutation_importance(
        best_pipe, X_val, y_val, n_repeats=5, random_state=42, n_jobs=-1
    )
    importances = pd.Series(result.importances_mean, index=X_val.columns)
    top_importances = importances.sort_values(ascending=False).head(20)
    print("Top Importances:")
    print(top_importances)

Computing Permutation Feature Importance for Two-Stage Classifier...
Top Importances:
flight_num_only                     0.055556
source_airport                      0.033333
airline_code                        0.027778
direction                           0.027778
route_airport_std                   0.005556
sin_hour                            0.005556
dew_point_c                         0.000000
temp_dew_spread                     0.000000
cloud_cover                         0.000000
visibility_bin                      0.000000
visibility_miles                    0.000000
wind_speed_kt                       0.000000
wind_direction_deg                  0.000000
scheduled_hour                      0.000000
is_estimated_missing                0.000000
flight_number                       0.000000
minutes_to_departure_at_snapshot    0.000000
scheduled_month                     0.000000
scheduled_dayofweek                 0.000000
cos_hour                            0.000000
dtype: float64

## Save best model artifact

In [13]:
import joblib

artifacts_dir = Path(".") / "artifacts"
artifacts_dir.mkdir(parents=True, exist_ok=True)

artifact_path = artifacts_dir / f"delay_model_twostage.joblib"
if best_model_name == "TwoStage":
    artifact = {
        "classifier": two_stage_clf,
        "regressor": two_stage_reg,
        "threshold": two_stage_threshold,
    }
    joblib.dump(artifact, artifact_path)
else:
    joblib.dump(best_pipe, artifact_path)

print("Artifact saved to:", artifact_path)

Artifact saved to: artifacts/delay_model_twostage.joblib


## Optuna tuning with TimeSeriesSplit

In [14]:
try:
    import optuna

    def objective(trial):
        params = {
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
            "max_depth": trial.suggest_int("max_depth", 3, 9),
            "max_iter": trial.suggest_int("max_iter", 100, 500),
            "random_state": 42,
        }
        
        tscv = TimeSeriesSplit(n_splits=3)
        mae_scores = []
        
        X_trainval_sorted = X_trainval.sort_index()
        y_trainval_sorted = y_trainval.sort_index()
        
        for train_idx, val_idx in tscv.split(X_trainval_sorted):
            X_tr, y_tr = X_trainval_sorted.iloc[train_idx], y_trainval_sorted.iloc[train_idx]
            X_va, y_va = X_trainval_sorted.iloc[val_idx], y_trainval_sorted.iloc[val_idx]
            
            # Using HistGradientBoostingRegressor for simplicity in tuning
            base_model = HistGradientBoostingRegressor(**params)
            model = get_log_regressor(base_model)
            
            pipe = Pipeline([
                ("preprocess", preprocess),
                ("model", model),
            ])
            
            pipe.fit(X_tr, y_tr)
            pred = pipe.predict(X_va)
            mae_scores.append(mean_absolute_error(y_va, pred))
            
        return np.mean(mae_scores)

    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=10) # 10 trials for demonstration

    print("Best parameters:", study.best_params)
except Exception as exc:
    print(f"Optuna not available or failed: {exc}")

[I 2026-05-08 13:00:55,944] A new study created in memory with name: no-name-9112ad04-e87a-4a48-903b-f0c7f46ed2f4
[I 2026-05-08 13:00:56,320] Trial 0 finished with value: 3.795870753378899 and parameters: {'learning_rate': 0.06261413219867129, 'max_depth': 8, 'max_iter': 363}. Best is trial 0 with value: 3.795870753378899.
[I 2026-05-08 13:00:56,749] Trial 1 finished with value: 3.637318548905675 and parameters: {'learning_rate': 0.011401626103252264, 'max_depth': 8, 'max_iter': 478}. Best is trial 1 with value: 3.637318548905675.
[I 2026-05-08 13:00:57,102] Trial 2 finished with value: 3.824505660320796 and parameters: {'learning_rate': 0.07658466245811103, 'max_depth': 6, 'max_iter': 387}. Best is trial 1 with value: 3.637318548905675.
[I 2026-05-08 13:00:57,336] Trial 3 finished with value: 3.6187274946279486 and parameters: {'learning_rate': 0.015509518261527017, 'max_depth': 6, 'max_iter': 235}. Best is trial 3 with value: 3.6187274946279486.
[I 2026-05-08 13:00:57,717] Trial 4 fi

Best parameters: {'learning_rate': 0.013754071823050357, 'max_depth': 4, 'max_iter': 126}
